In [ ]:
import numpy as np
import pandas as pd

# Load dataset
df = pd.read_csv("train.csv")

# Preview top 5 rows
df.head()


In [ ]:
# Displays total rows, columns, data types, and non-null counts
print("--- Dataset Info ---")
df.info()

print("\n--- Shape (Rows, Columns) ---")
print(df.shape)

In [ ]:
# Numerical columns summary (mean, min, max, quartiles)
print("--- Numerical Features Summary ---")
df.describe()

In [ ]:
# Identify exact counts of missing values
print("--- Missing Values Per Column ---")
missing = df.isnull().sum()
print(missing[missing > 0])

print("\n--- Categorical Features Summary ---")
df.describe(include=["O"])

### Titanic Dataset: Exploratory Data Analysis Summary

* **Dataset Dimensions:** The dataset consists of **891 rows** and **12 columns**, representing individual passenger records.
* **Missing Data:** Significant missing values exist in `Cabin` (687 missing, ~77%) and `Age` (177 missing, ~20%), with minor missing values in `Embarked` (2 missing).
* **Numerical Features:** Includes 7 numerical attributes (`PassengerId`, `Survived`, `Pclass`, `Age`, `SibSp`, `Parch`, `Fare`).
* **Categorical Features:** Includes 5 non-numeric attributes (`Name`, `Sex`, `Ticket`, `Cabin`, `Embarked`).
* **Key Takeaway:** Prior to modeling, `Cabin` may need to be dropped due to missingness, `Age` requires imputation, and categorical variables like `Sex` and `Embarked` will need encoding.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Load data again to work fresh or continue from previous df
df = pd.read_csv("train.csv")

# 1. Fill missing 'Age' with the median (robust against extreme age outliers)
df['Age'].fillna(df['Age'].median(), inplace=True)

# 2. Fill missing 'Embarked' with the mode (most frequent port: 'S')
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# 3. Drop 'Cabin' because ~77% of its data is missing (too sparse to impute accurately)
df.drop(columns=['Cabin'], inplace=True)

# Verify no missing values remain in core columns
print("Remaining missing values:")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['Age'], kde=True, bins=30, color='skyblue')
plt.title('Distribution of Passenger Ages (Post-Imputation)')
plt.xlabel('Age')
plt.ylabel('Count')
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df['Fare'], color='salmon')
plt.title('Boxplot of Passenger Fares (Outlier Detection)')
plt.xlabel('Fare ($)')
plt.show()

# Quick print of top extreme fare outliers
print("Top 5 Highest Fares paid:")
print(df['Fare'].nlargest(5))

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(x='Sex', y='Survived', data=df, palette='Set2', ci=None)
plt.title('Survival Rate by Gender')
plt.ylabel('Survival Probability')
plt.xlabel('Sex')
plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
# Filter for numeric columns only for correlation
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title('Numeric Feature Correlation Heatmap')
plt.show()

### Data Cleaning Justification
* **Age:** Imputed using the **median age (~28)** rather than the mean to prevent skewing caused by infant and elderly age extremities.
* **Embarked:** Filled 2 missing values with the **mode ('S' - Southampton)**, as over 70% of passengers boarded from there.
* **Cabin:** Dropped the feature entirely. With over 77% missing values, filling it would introduce excessive artificial noise.

---

### Key Insight: Which feature most affects survival, and why?
Based on our exploratory visualizations, **`Sex` (Gender)** is the single strongest factor influencing survival. 

1. **Survival Probability:** Females had a **~74% survival rate**, compared to only **~19% for males**.
2. **Historical & Domain Context:** This stark divide is heavily explained by the maritime protocol *"women and children first"* during the loading of lifeboats.
3. **Secondary Factor (`Pclass`):** Passenger Class was the second strongest predictor; 1st-class passengers had significantly higher survival rates than 3rd-class passengers due to proximity to the upper decks where lifeboats were staged.